# Imports

In [2]:
import os
import sys
import glob
import tempfile
import pandas as pd
from pcapflower import convert_pcap_to_csv

# Inputs

In [2]:
DATA_ROOT = "../data"

# Folder names to skip (relative to DATA_ROOT, e.g. ["benign", "dos_attack"])
EXCLUDE_FOLDERS = ["CIC_IIoT_dataset_2025"]

# Preprocessing

In [3]:
def find_pcap_folders(root_dir, exclude=None):
    """Return every folder under root_dir that contains at least one PCAP file."""
    exclude = set(exclude or [])
    found = []
    for dirpath, _, filenames in os.walk(root_dir):
        if os.path.basename(dirpath) in exclude:
            continue
        if any(f.endswith((".pcap", ".pcapng")) for f in filenames):
            found.append(dirpath)
    return sorted(found)


def generate_merged_csv_from_pcaps(pcap_dir, output_filename):
    pcap_files = sorted(
        glob.glob(os.path.join(pcap_dir, "*.pcap")) +
        glob.glob(os.path.join(pcap_dir, "*.pcapng"))
    )

    if not pcap_files:
        raise FileNotFoundError(f"No PCAP files found in: {pcap_dir}")

    label = os.path.basename(pcap_dir)
    print(f"[{label}] {len(pcap_files)} PCAP file(s)")

    output_path = os.path.join(pcap_dir, output_filename)
    total_flows = 0
    first_write = True

    with tempfile.TemporaryDirectory(dir=pcap_dir) as tmp_dir:
        for i, pcap_path in enumerate(pcap_files, 1):
            pcap_name = os.path.basename(pcap_path)
            print(f"  [{i}/{len(pcap_files)}] {pcap_name}")

            tmp_csv = os.path.join(tmp_dir, f"{pcap_name}.csv")
            n_flows = convert_pcap_to_csv(pcap_path, tmp_csv)
            print(f"    → {n_flows} flows")

            if n_flows > 0:
                for chunk in pd.read_csv(tmp_csv, chunksize=50_000):
                    chunk["label"] = label
                    chunk.to_csv(output_path, mode="w" if first_write else "a",
                                 index=False, header=first_write)
                    total_flows += len(chunk)
                    first_write = False

    if total_flows == 0:
        raise RuntimeError(f"No flows extracted from: {pcap_dir}")

    print(f"  ✓ {output_path}  ({total_flows} flows)")
    return output_path

In [4]:
pcap_folders = find_pcap_folders(DATA_ROOT, exclude=EXCLUDE_FOLDERS)
print(f"Found {len(pcap_folders)} folder(s) with PCAPs\n")

for folder_path in pcap_folders:
    folder_name = os.path.basename(folder_path)
    generate_merged_csv_from_pcaps(folder_path, f"merged_{folder_name}.csv")

Found 35 folder(s) with PCAPs

[CTU-Honeypot-Capture-4-1] 2 PCAP file(s)
  [1/2] 2018-09-14-13-40-25-Philips-Hue-Bridge.pcap
    → 829 flows
  [2/2] 2018-10-25-14-06-32-192.168.1.132.pcap
    → 721 flows
  ✓ ../data/raw/Aposemat_IoT_23/iot_23_datasets_full/opt/Malware-Project/BigDataset/IoTScenarios/CTU-Honeypot-Capture-4-1/merged_CTU-Honeypot-Capture-4-1.csv  (1550 flows)
[CTU-Honeypot-Capture-5-1] 1 PCAP file(s)
  [1/1] 2018-09-21-capture.pcap
    → 1826 flows
  ✓ ../data/raw/Aposemat_IoT_23/iot_23_datasets_full/opt/Malware-Project/BigDataset/IoTScenarios/CTU-Honeypot-Capture-5-1/merged_CTU-Honeypot-Capture-5-1.csv  (1826 flows)
[Somfy-01] 1 PCAP file(s)
  [1/1] 2019-07-03-15-15-47-first_start_somfy_gateway.pcap
    → 37 flows
  ✓ ../data/raw/Aposemat_IoT_23/iot_23_datasets_full/opt/Malware-Project/BigDataset/IoTScenarios/CTU-Honeypot-Capture-7-1/Somfy-01/merged_Somfy-01.csv  (37 flows)
[Somfy-02] 1 PCAP file(s)
  [1/1] 2019-07-03-16-41-09-192.168.1.158.pcap
    → 56 flows
  ✓ ../dat

KeyboardInterrupt: 